# AG_PRAXIS NB06d — DoS and DDoS Pair Separability Under Window Aggregation

A model that reads one record at a time separates the DoS and DDoS forms of the same
protocol almost perfectly. The same encoder, reading fifty consecutive records and
summarising them before it decides, does not. Most of what it gets wrong on those
classes is putting one member of a pair into the other.

So the question here is narrow. For each of the four DoS and DDoS pairs, is the
distinction between the two members present in a single record, and does averaging
across fifty records destroy it? If a feature separates the two per record and stops
separating once averaged, then aggregation is where the distinction goes, and that is
measurable without training anything.

All four pairs are treated identically, and the UDP pair matters as much as the others.
This notebook also counts, from the predictions the window model saved when it ran,
which pairs it actually confuses. If one pair is confused far less than the rest, then
any mechanism offered for the other three has to leave that one alone. A mechanism that
predicts damage everywhere would explain nothing.

Nothing is trained. The windows were cut and saved earlier, and this reads them back.

The usual first cell: Drive, the repository, and the commit this ran at.

In [ ]:
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")

What this reads and where it writes. Three things come off Drive: the manifest that
records how the windows were cut, the saved test windows themselves, and the predictions
the window model wrote when it was scored. The four pairs are named here rather than
found, because which classes form a DoS and DDoS pair is a property of the benchmark's
own naming and not something to be discovered from the data.

In [ ]:
import json

import numpy as np
import pandas as pd

from src import eda
from src import inventory as inv

CFG = inv.load_config(REPO_ROOT)
SEED = CFG["seed"]
WINDOW = int(CFG["sequence"]["window"])
STRIDE = int(CFG["sequence"]["stride"])
ARTIFACTS = Path(CFG["paths"]["artifacts"])

FAST = os.environ.get("FAST", "0") == "1"
OUT_DIR = ARTIFACTS / ("NB06d_fast" if FAST else "NB06d")
OUT_DIR.mkdir(parents=True, exist_ok=True)

if IN_COLAB and not ARTIFACTS.exists():
    raise FileNotFoundError(f"{ARTIFACTS} does not exist, so nothing this notebook writes survives.")


def first_existing(candidates, what):
    found = next((Path(p) for p in candidates if Path(p).exists()), None)
    if found is None:
        raise FileNotFoundError(f"{what} not found. Looked in: {[str(p) for p in candidates]}")
    return found


MANIFEST = json.loads(first_existing(
    [REPO_ROOT / "data" / "processed" / "NB04_manifest.json",
     ARTIFACTS / "NB04" / "NB04_manifest.json"], "the window manifest").read_text())
ARRAY_DIR = first_existing([ARTIFACTS / "NB04"], "the saved window arrays")
PREDICTIONS_DIR = first_existing(
    [REPO_ROOT / "data" / "processed" / "NB06" / "sequence_cnn_lstm_19class",
     ARTIFACTS / "NB06" / "sequence_cnn_lstm_19class"],
    "the window model's saved predictions")

PAIRS = [("DoS-ICMP", "DDoS-ICMP"), ("DoS-TCP", "DDoS-TCP"),
         ("DoS-SYN", "DDoS-SYN"), ("DoS-UDP", "DDoS-UDP")]

print(f"windows     : {ARRAY_DIR}")
print(f"predictions : {PREDICTIONS_DIR}")
print(f"writing to  : {OUT_DIR}")
print(f"window {WINDOW}, stride {STRIDE}, seed {SEED}")
print(f"pairs       : {', '.join(a + ' / ' + b for a, b in PAIRS)}")

Seeding, and what the session is running on. Nothing here is trained and nothing
needs an accelerator, but the versions still go into what this writes, because a run
whose environment is remembered rather than recorded cannot be read back later against
the machine that produced it.

In [ ]:
import random

import keras
import tensorflow as tf

random.seed(SEED)
np.random.seed(SEED)


def accelerator():
    """What this ran on, by name, so a wall time can be read against the hardware."""
    devices = tf.config.list_physical_devices("GPU")
    if not devices:
        return "cpu"
    named = []
    for device in devices:
        details = tf.config.experimental.get_device_details(device)
        named.append(str(details.get("device_name", device.name)))
    return ", ".join(named)


ENVIRONMENT = {
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "backend": keras.backend.backend(),
    "accelerator": accelerator(),
    "run_date": RUN_DATE,
}

print(f"seeded with : {SEED}")
print(f"environment : {ENVIRONMENT}")

Reading the test windows. Each one is fifty consecutive records of forty-four
features, standardised when they were built, and carrying an integer class code.

Nothing in the array says which name each code stands for. The manifest does say how many
test windows each named class has, and those counts differ from class to class, so
checking the count under each code against the count recorded for the name in that
position settles the mapping instead of assuming it. If any class had been written in a
different position the check fails rather than mislabelling every table below.

In [ ]:
CLASSES = sorted(MANIFEST["arrays"]["sequences_test"]["by_class"])
RECORDED_COUNTS = {k: int(v) for k, v in MANIFEST["arrays"]["sequences_test"]["by_class"].items()}

with np.load(ARRAY_DIR / "sequences_test.npz", allow_pickle=False) as npz:
    FEATURES = [str(v) for v in npz["features"]]
    X = npz["X"]
    Y = npz["y"].astype("int64")
    FILE_WINDOW = int(npz["window"])
    FILE_STRIDE = int(npz["stride"])

assert X.ndim == 3 and len(X) == len(Y), f"windows {X.shape} against {len(Y)} labels"
assert tuple(X.shape[1:]) == (WINDOW, len(FEATURES)), f"unexpected window shape {X.shape}"
assert (FILE_WINDOW, FILE_STRIDE) == (WINDOW, STRIDE), "the file was cut at another window"
assert len(FEATURES) == 44 and len(CLASSES) == 19

OBSERVED_COUNTS = {CLASSES[c]: int((Y == c).sum()) for c in range(len(CLASSES))}
MISMATCHED = {name: (OBSERVED_COUNTS[name], RECORDED_COUNTS[name])
              for name in CLASSES if OBSERVED_COUNTS[name] != RECORDED_COUNTS[name]}
assert not MISMATCHED, f"class codes do not carry the names assumed: {MISMATCHED}"

CODE = {name: i for i, name in enumerate(CLASSES)}
for a, b in PAIRS:
    assert a in CODE and b in CODE, f"{a} or {b} is not one of the classes in the file"

print(f"{len(Y):,} test windows of {WINDOW} records over {len(FEATURES)} features")
print("class codes agree with the recorded per-class counts, so the names below are the "
      "names the arrays carry")
print()
print("windows in each pair")
for a, b in PAIRS:
    print(f"  {a:<9} {OBSERVED_COUNTS[a]:>7,}      {b:<10} {OBSERVED_COUNTS[b]:>7,}"
          f"      total {OBSERVED_COUNTS[a] + OBSERVED_COUNTS[b]:>7,}")

How separability is measured. For one feature and two classes, the area under the ROC
curve is the chance that a value drawn from one class comes out above a value drawn from
the other, with a tie counted as half. At 0.5 the feature says nothing about which class a
value came from, and at 1.0 it says everything. A feature that runs the other way separates
just as well, so the better of the two directions is taken and every figure below sits
between 0.5 and 1.0.

Each pair gets three readings of the same forty-four features, over the same windows.
The first pools the individual records and asks how well one record separates the pair.
The second takes each window's mean and asks the same question of that, which is what
survives aggregation. The third takes each window's standard deviation, so that if the
two members differ in how much a feature moves inside a window rather than in its level,
the difference is still visible somewhere.

In [ ]:
def pair_table(a, b):
    """Three separability readings for one pair, all over the same windows."""
    keep = np.isin(Y, [CODE[a], CODE[b]])
    windows, labels = X[keep], Y[keep]
    is_a = labels == CODE[a]

    records = windows.reshape(-1, len(FEATURES))
    record_is_a = np.repeat(is_a, WINDOW)
    means = windows.mean(axis=1)
    spreads = windows.std(axis=1)

    frame = pd.DataFrame({
        "feature": FEATURES,
        "auc_record": eda.auc_by_column(records[record_is_a], records[~record_is_a]),
        "auc_window_mean": eda.auc_by_column(means[is_a], means[~is_a]),
        "auc_window_sd": eda.auc_by_column(spreads[is_a], spreads[~is_a]),
        "sd_in_window_a": spreads[is_a].mean(axis=0),
        "sd_in_window_b": spreads[~is_a].mean(axis=0),
    })
    frame["lost_to_averaging"] = frame["auc_record"] - frame["auc_window_mean"]
    frame["recovered_by_spread"] = frame["auc_window_sd"] - frame["auc_window_mean"]
    # A member whose feature never moves inside a window gives a zero denominator, and
    # a ratio of the two spreads is undefined there rather than infinite.
    numerator = frame["sd_in_window_a"].to_numpy(dtype="float64")
    denominator = frame["sd_in_window_b"].to_numpy(dtype="float64")
    safe = denominator > 0
    frame["sd_ratio"] = np.where(safe, numerator / np.where(safe, denominator, 1.0), np.nan)

    numeric = [column for column in frame.columns if column != "feature"]
    assert all(pd.api.types.is_numeric_dtype(frame[column]) for column in numeric), (
        f"non-numeric column in the {a} against {b} table: "
        f"{[c for c in numeric if not pd.api.types.is_numeric_dtype(frame[c])]}")
    return frame.sort_values("lost_to_averaging", ascending=False).reset_index(drop=True)


TABLES = {}
for PAIR_A, PAIR_B in PAIRS:
    TABLES[(PAIR_A, PAIR_B)] = pair_table(PAIR_A, PAIR_B)
    print(f"{PAIR_A} against {PAIR_B}: {len(FEATURES)} features read three ways")

The two columns to read against each other are `auc_record` and `auc_window_mean`. A
feature high in the first and near 0.5 in the second is one the pair can be told apart by
in a single record and cannot be told apart by once fifty of them are averaged.

In [ ]:
SHOW = ["feature", "auc_record", "auc_window_mean", "lost_to_averaging",
        "auc_window_sd", "recovered_by_spread"]
STRONG = 0.70

SUMMARY_ROWS = []
for (a, b), frame in TABLES.items():
    strong = frame[frame["auc_record"] >= STRONG]
    survived = int((strong["auc_window_mean"] >= STRONG).sum())
    row = {
        "pair": f"{a} / {b}",
        "separating per record": len(strong),
        "still separating on the mean": survived,
        "mean AUC lost": float(frame["lost_to_averaging"].mean()),
        "features losing 0.10 or more": int((frame["lost_to_averaging"] >= 0.10).sum()),
        "best AUC on the mean": float(frame["auc_window_mean"].max()),
        "best AUC on the spread": float(frame["auc_window_sd"].max()),
    }
    SUMMARY_ROWS.append(row)

    print("=" * 92)
    print(f"{a} against {b}")
    print("=" * 92)
    print(f"  features separating the pair per record at AUC {STRONG:.2f} or better : "
          f"{len(strong)} of {len(FEATURES)}")
    print(f"  of those, still separating at {STRONG:.2f} on the window mean         : {survived}")
    print(f"  mean AUC lost to averaging, across all {len(FEATURES)} features        : "
          f"{row['mean AUC lost']:.4f}")
    print(f"  best any feature manages on the window mean                        : "
          f"{row['best AUC on the mean']:.4f}")
    print()
    print(frame.head(10)[SHOW].to_string(index=False, float_format=lambda v: f"{v:8.4f}"))
    print()

SUMMARY = pd.DataFrame(SUMMARY_ROWS)

Now the other half of the question. Two classes can carry the same average and still
differ in how much a feature moves from record to record inside a window. Averaging
removes exactly that: the mean of fifty records keeps the level and throws away the
variation around it. So for each pair I count the features that have stopped separating
on the window mean, and ask how many of them separate on the window's standard deviation
instead. Those are the features whose distinction is a difference in variability rather
than in level.

The two cut-offs are chosen here and stated so they can be argued with: below 0.55 counts
as not separating on level, and 0.70 or above counts as separating on spread.

In [ ]:
LEVEL_CUT = 0.55
SPREAD_CUT = 0.70

SPREAD_ROWS = []
for (a, b), frame in TABLES.items():
    quiet = frame["auc_window_mean"] < LEVEL_CUT
    loud = quiet & (frame["auc_window_sd"] >= SPREAD_CUT)
    SPREAD_ROWS.append({
        "pair": f"{a} / {b}",
        "flat on the window mean": int(quiet.sum()),
        "of those, separating on spread": int(loud.sum()),
        "best spread AUC among them": (float(frame.loc[quiet, "auc_window_sd"].max())
                                       if bool(quiet.any()) else float("nan")),
        "median sd ratio, first over second": float(frame["sd_ratio"].median(skipna=True)),
    })
    if bool(loud.any()):
        print(f"{a} against {b}: features flat on the mean and separating on the spread")
        print(frame.loc[loud, ["feature", "auc_window_mean", "auc_window_sd",
                               "sd_in_window_a", "sd_in_window_b", "sd_ratio"]]
              .to_string(index=False, float_format=lambda v: f"{v:8.4f}"))
        print()

SPREAD_TABLE = pd.DataFrame(SPREAD_ROWS)
print("the same counts, side by side")
print(SPREAD_TABLE.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))
print()
print("The sd ratio is the mean within-window standard deviation of the first member "
      "divided by that of the second, averaged over features. Above 1 the first member "
      "moves more inside a window than the second.")

One comparison across the four pairs. If losing separability to averaging is what
damages a pair, then the pair that loses the most should be the pair the window model
confuses most, and the pair that loses the least should be the one it confuses least.

The confusion is not taken on trust here. The model wrote down its predictions when it was
scored, so for each pair I take the windows whose true class is one of the two members and
count how many were predicted as the other member. That is a rate this notebook computes
from the saved arrays, and the same per-class count check is applied to those arrays first
so the class codes in them are known to carry the same names.

In [ ]:
PRED_TRUE = np.load(PREDICTIONS_DIR / "y_true.npy").astype("int64")
PRED = np.load(PREDICTIONS_DIR / "y_pred.npy").astype("int64")

assert PRED_TRUE.shape == PRED.shape, f"{PRED_TRUE.shape} against {PRED.shape}"
PRED_COUNTS = {CLASSES[c]: int((PRED_TRUE == c).sum()) for c in range(len(CLASSES))}
PRED_MISMATCH = {name: (PRED_COUNTS[name], RECORDED_COUNTS[name])
                 for name in CLASSES if PRED_COUNTS[name] != RECORDED_COUNTS[name]}
assert not PRED_MISMATCH, f"the saved predictions do not carry these classes: {PRED_MISMATCH}"

CONFUSION_ROWS = []
for a, b in PAIRS:
    inside = np.isin(PRED_TRUE, [CODE[a], CODE[b]])
    true_side, predicted = PRED_TRUE[inside], PRED[inside]
    a_to_b = int(((true_side == CODE[a]) & (predicted == CODE[b])).sum())
    b_to_a = int(((true_side == CODE[b]) & (predicted == CODE[a])).sum())
    n_inside = int(inside.sum())
    CONFUSION_ROWS.append({
        "pair": f"{a} / {b}",
        "windows in the pair": n_inside,
        "swapped for the partner": a_to_b + b_to_a,
        "confusion rate": (a_to_b + b_to_a) / n_inside if n_inside else float("nan"),
        "first into second": a_to_b,
        "second into first": b_to_a,
    })
CONFUSION = pd.DataFrame(CONFUSION_ROWS)

CROSS = SUMMARY.merge(CONFUSION, on="pair")
CROSS = CROSS.sort_values("mean AUC lost", ascending=False).reset_index(drop=True)

ORDER_BY_LOSS = list(CROSS["pair"])
ORDER_BY_CONFUSION = list(CONFUSION.sort_values("confusion rate", ascending=False)["pair"])
ORDERS_AGREE = ORDER_BY_LOSS == ORDER_BY_CONFUSION

print(CROSS[["pair", "mean AUC lost", "features losing 0.10 or more",
             "best AUC on the mean", "confusion rate", "swapped for the partner"]]
      .to_string(index=False, float_format=lambda v: f"{v:8.4f}"))
print()
print(f"ranked by separability lost   : {' > '.join(ORDER_BY_LOSS)}")
print(f"ranked by how often confused  : {' > '.join(ORDER_BY_CONFUSION)}")
print(f"the two orderings {'agree' if ORDERS_AGREE else 'do not agree'}")

Two figures. The first puts every feature of every pair as one point, per-record
separability against window-mean separability, so a point far below the diagonal is a
feature that survives in one record and not in fifty. The second reduces each pair to how
much it loses, so the four can be read side by side.

In [ ]:
import matplotlib

if not IN_COLAB:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt

FIG_ONE = OUT_DIR / "NB06d_record_against_window_mean.png"

fig, axes = plt.subplots(2, 2, figsize=(10.5, 10), sharex=True, sharey=True)
for axis, ((a, b), frame) in zip(axes.ravel(), TABLES.items()):
    axis.plot([0.5, 1.0], [0.5, 1.0], color="0.6", linewidth=1, zorder=1)
    axis.scatter(frame["auc_record"], frame["auc_window_mean"], s=26, zorder=2,
                 color="#1f77b4", edgecolor="white", linewidth=0.5)
    for _, line in frame.head(3).iterrows():
        axis.annotate(line["feature"], (line["auc_record"], line["auc_window_mean"]),
                      textcoords="offset points", xytext=(5, -9), fontsize=7.5, color="0.25")
    axis.set_title(f"{a} against {b}", fontsize=11)
    axis.set_xlim(0.48, 1.02)
    axis.set_ylim(0.48, 1.02)
    axis.grid(alpha=0.25, linewidth=0.6)
for axis in axes[-1]:
    axis.set_xlabel("AUC on one record")
for axis in axes[:, 0]:
    axis.set_ylabel("AUC on the window mean")
fig.suptitle("Separability of each DoS and DDoS pair, per record against per window mean",
             fontsize=12.5)
fig.tight_layout()
fig.savefig(FIG_ONE, dpi=150)
plt.close(fig)
print(f"wrote {FIG_ONE}")

In [ ]:
FIG_TWO = OUT_DIR / "NB06d_separability_lost_by_pair.png"

ordered = CROSS.sort_values("mean AUC lost", ascending=False)
positions = np.arange(len(ordered))

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4.6))
left.bar(positions, ordered["mean AUC lost"], color="#1f77b4", width=0.62)
left.set_xticks(positions)
left.set_xticklabels([p.replace(" / ", "\n") for p in ordered["pair"]], fontsize=9)
left.set_ylabel("mean AUC lost to averaging")
left.set_title("Separability lost, averaged over 44 features", fontsize=11)
left.grid(axis="y", alpha=0.25, linewidth=0.6)

right.bar(positions, ordered["confusion rate"], color="#d62728", width=0.62)
right.set_xticks(positions)
right.set_xticklabels([p.replace(" / ", "\n") for p in ordered["pair"]], fontsize=9)
right.set_ylabel("share of the pair's windows swapped")
right.set_title("How often the window model swaps the two members", fontsize=11)
right.grid(axis="y", alpha=0.25, linewidth=0.6)

fig.suptitle("The four pairs, ordered by how much averaging costs them", fontsize=12.5)
fig.tight_layout()
fig.savefig(FIG_TWO, dpi=150)
plt.close(fig)
print(f"wrote {FIG_TWO}")

One property of the data bears on all of the above and can be read straight out of the
values. Every row in these files is already a summary of several packets rather than a
single packet, so a window of fifty rows is not fifty packets, and how much traffic it
covers depends on how many packets went into each row.

Two readings of that are taken here, and the values have to be put back on their original
scale first, which the manifest's scaler allows. The first is the `Number` column read
directly, per class. The second is indirect and independent of what any column is named:
a column recording whether a packet carried a particular flag or ran over a particular
protocol, averaged over a fixed number of packets, can only take values that are multiples
of one over that number, so reading the distinct values back bounds the packet count from
below. That second reading only says anything for a class whose windows mix packets of
different kinds. Where every such column sits at a single value throughout, it says
nothing, and it reports nothing rather than defaulting to the smallest candidate.

In [ ]:
# The scaler is recorded per feature name rather than as a bare list, so the columns
# are looked up by name and a feature the scaler does not cover fails here by name.
UNSCALED = [name for name in FEATURES
            if name not in MANIFEST["scaler"]["mean"] or name not in MANIFEST["scaler"]["scale"]]
assert not UNSCALED, f"the scaler does not cover {UNSCALED}"

SCALER_MEAN = np.asarray([MANIFEST["scaler"]["mean"][name] for name in FEATURES], dtype="float64")
SCALER_SCALE = np.asarray([MANIFEST["scaler"]["scale"][name] for name in FEATURES], dtype="float64")
assert len(SCALER_MEAN) == len(FEATURES) and len(SCALER_SCALE) == len(FEATURES)

INDICATOR_NAMES = [name for name in FEATURES
                   if name.endswith("_flag_number")
                   or name in {"HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC",
                               "TCP", "UDP", "DHCP", "ARP", "ICMP", "IGMP", "IPv", "LLC"}]
INDICATOR_INDEX = [FEATURES.index(name) for name in INDICATOR_NAMES]
NUMBER_INDEX = FEATURES.index("Number") if "Number" in FEATURES else None
WINDOWS_SAMPLED = 200


def unscale(block, columns):
    """Standardised values back to what the files hold."""
    return block * SCALER_SCALE[columns] + SCALER_MEAN[columns]


def packets_per_row(name):
    """Both readings of how many packets went into one row of this class."""
    where = np.flatnonzero(Y == CODE[name])
    if not len(where):
        return {}
    picker = np.random.default_rng(SEED)
    if len(where) > WINDOWS_SAMPLED:
        where = np.sort(picker.choice(where, size=WINDOWS_SAMPLED, replace=False))
    block = X[where][:, :, INDICATOR_INDEX].reshape(-1, len(INDICATOR_INDEX))
    raw = unscale(block, INDICATOR_INDEX)
    # A column that never moves for this class is consistent with any packet count and
    # would let the smallest candidate through on no evidence at all.
    informative = raw[:, raw.std(axis=0) > 0]
    reading = {
        "windows read": int(len(where)),
        "indicator columns that vary": int(informative.shape[1]) if informative.size else 0,
        "indicator reading": (eda.smallest_consistent_denominator(informative)
                              if informative.size else None),
    }
    if NUMBER_INDEX is not None:
        values = unscale(X[where][:, :, NUMBER_INDEX].ravel(), NUMBER_INDEX)
        reading["Number min"] = float(values.min())
        reading["Number median"] = float(np.median(values))
        reading["Number max"] = float(values.max())
    return reading


PACKETS = pd.DataFrame([{"class": name, **packets_per_row(name)} for name in CLASSES])
PACKETS["packets per row"] = PACKETS["Number median"].round().astype("float64")

print(PACKETS.to_string(index=False, float_format=lambda v: f"{v:10.3f}"))
print()

PAIR_MEMBERS = sorted({name for pair in PAIRS for name in pair})
INSIDE = sorted(PACKETS.loc[PACKETS["class"].isin(PAIR_MEMBERS),
                            "packets per row"].dropna().unique().tolist())
OUTSIDE = sorted(PACKETS.loc[~PACKETS["class"].isin(PAIR_MEMBERS),
                             "packets per row"].dropna().unique().tolist())
SPLIT_ON_PAIRS = bool(INSIDE) and bool(OUTSIDE) and not set(INSIDE) & set(OUTSIDE)

print(f"rounded Number median, the eight DoS and DDoS classes : {INSIDE}")
print(f"rounded Number median, the other eleven classes       : {OUTSIDE}")
print()
if SPLIT_ON_PAIRS:
    print("The two groups take different values and do not overlap, so a row means a "
          "different quantity of traffic on either side of that division, and the eight "
          "classes in these four pairs are all on one side of it. A window of 50 records "
          "is 50 rows either way, and not the same amount of traffic either way.")
else:
    print("The two groups do not separate cleanly on this reading, so nothing is claimed "
          "here about a row meaning different amounts of traffic for different classes.")

Writing it down, so the tables can be read without running any of this again.

In [ ]:
DOCUMENT = {
    "generated_by": "AG_PRAXIS_NB06d_dosddos_pair_diagnosis.ipynb",
    "generated_on": RUN_DATE, "git_sha": GIT_SHA, "git_dirty": GIT_DIRTY,
    "seed": SEED, "is_fast_pass": bool(FAST), "environment": ENVIRONMENT,
    "trained": "nothing",
    "read": {"windows": str(ARRAY_DIR / "sequences_test.npz"),
             "predictions": str(PREDICTIONS_DIR)},
    "window": WINDOW, "stride": STRIDE,
    "features": FEATURES, "classes": CLASSES,
    "test_windows": int(len(Y)),
    "separability": {
        "definition": "best-direction area under the ROC curve between the two members "
                      "of a pair, per feature; 0.5 is no separation and 1.0 is complete",
        "readings": {"auc_record": "pooled individual records",
                     "auc_window_mean": "the mean of the 50 records of a window",
                     "auc_window_sd": "the standard deviation across those 50 records"},
        "cut_offs": {"strong": STRONG, "flat_on_level": LEVEL_CUT, "separates_on_spread": SPREAD_CUT},
    },
    "pairs": {f"{a}|{b}": TABLES[(a, b)].to_dict(orient="records") for a, b in PAIRS},
    "summary": SUMMARY.to_dict(orient="records"),
    "spread": SPREAD_TABLE.to_dict(orient="records"),
    "confusion_from_saved_predictions": CONFUSION.to_dict(orient="records"),
    "cross_pair": CROSS.to_dict(orient="records"),
    "ordering": {"by_separability_lost": ORDER_BY_LOSS,
                 "by_confusion_rate": ORDER_BY_CONFUSION,
                 "agree": bool(ORDERS_AGREE)},
    "packets_per_row": PACKETS.to_dict(orient="records"),
    "packets_per_row_method": {
        "Number": "the Number column read back on its original scale, per class",
        "indicator": "the smallest packet count for which every value of every per-packet "
                     "indicator column that varies is a multiple of one over that count",
        "separates_the_pair_classes": bool(SPLIT_ON_PAIRS),
    },
    "figures": [FIG_ONE.name, FIG_TWO.name],
}

OUT_FILE = OUT_DIR / "pair_separability.json"
OUT_FILE.write_text(json.dumps(DOCUMENT, indent=1, default=float))
print(f"wrote {OUT_FILE}")
print(f"      {len(DOCUMENT['pairs'])} pairs, {len(FEATURES)} features each")

The ledger entry for this run.

In [ ]:
status = ("DO NOT ENTER, fast pass" if FAST else
          "reported result, working tree dirty" if GIT_DIRTY else
          "reported result, diagnostic; no model trained and no hypothesis tested")

worst = CROSS.iloc[0]
best = CROSS.iloc[-1]

ledger = f"""
### NB06d — DoS and DDoS pair separability under window aggregation ({RUN_DATE})

| field | value |
|---|---|
| notebook | AG_PRAXIS_NB06d_dosddos_pair_diagnosis.ipynb |
| run date | {RUN_DATE} |
| git sha | {GIT_SHA}{" (working tree dirty)" if GIT_DIRTY else ""} |
| seed | {SEED} |
| status | {status} |
| environment | {ENVIRONMENT} |
| trained | nothing; reads saved windows and saved predictions |
| scored on | {len(Y):,} test windows of {WINDOW} records over {len(FEATURES)} features |
| pairs | {", ".join(a + " / " + b for a, b in PAIRS)} |
| measure | best-direction AUC per feature, per record against per window mean |
| most lost | {worst['pair']}, mean AUC lost {worst['mean AUC lost']:.4f}, {int(worst['features losing 0.10 or more'])} features losing 0.10 or more, confusion rate {worst['confusion rate']:.4f} |
| least lost | {best['pair']}, mean AUC lost {best['mean AUC lost']:.4f}, {int(best['features losing 0.10 or more'])} features losing 0.10 or more, confusion rate {best['confusion rate']:.4f} |
| ranked by separability lost | {" > ".join(ORDER_BY_LOSS)} |
| ranked by confusion rate | {" > ".join(ORDER_BY_CONFUSION)} |
| orderings agree | {ORDERS_AGREE} |
| packets per row, the eight pair classes | {INSIDE} |
| packets per row, the other eleven | {OUTSIDE} |
| the two groups separate | {SPLIT_ON_PAIRS} |
| artefacts | {OUT_DIR}, holding {OUT_FILE.name}, {FIG_ONE.name}, {FIG_TWO.name} |
"""

print(ledger)